# Part II. Quantum states, Hamiltonians, and their optimization

In this exercise, we leverage PEPS to build & probe two dimensional quantum states.
Furthermore, we show how to optimize iPEPS targeting ground state of interacting Hamiltonian in the thermodynamic limit.

**Before you start**

Get the [standard tutorial dependencies](../README.md#installation).
Run cells from top to bottom. Four exercise groups concern the singlet,
projection, sublattice rotation, and double layers. All plotting, CTMRG,
density matrices, and transfer contractions are provided.
Replace each `raise NotImplementedError(...)` with your implementation, then rerun
that cell and its checks. The companion solution provides complete implementations.

## II.0 Pre-requisites

We will make use of CTMRG algorithm, here in a compact form

In [ ]:
# Provided: imports and numerical conventions.
import torch
torch.set_default_dtype(torch.float64)
torch.set_default_device("cpu")

def ctmrg(
    A: torch.Tensor,
    chi: int,
    ctm_max_iter=100,
    ctm_conv_tol=1.0e-8,
    callback=None,
    raise_on_nonconvergence=False,
):
    """Contract a PEPS norm with real C4v symmetry, retaining at most chi states.

    A has legs (up, left, down, right, physical). Returns C, T.
    callback(i, residual, finished, A, C, T) monitors each update.
    finished marks convergence or the final allowed iteration.
    Signed corner weights are retained; convergence compares their magnitudes.
    """
    assert A.ndim == 5, "A must be a 5-dimensional tensor."

    # Pair each ket/bra virtual leg to form the rank-four norm tensor.
    shape = tuple(d * d for d in A.shape[:4])
    E = torch.einsum("uldrp,ULDRp->uUlLdDrR", A, A.conj()).reshape(shape)

    # Trace the missing ket/bra pairs to initialize a free virtual boundary.
    C = torch.einsum("uldrp,ulDRp->dDrR", A, A.conj()).reshape(A.shape[2]**2, A.shape[3]**2)
    #
    #  --A--
    #   / \p   = 0--T--1   with usual index ordering for T [left,right,down]
    #    --A--      |
    #     /         2
    #
    T = torch.einsum("uldrp,uLDRp->lLrRdD", A, A.conj()).reshape(A.shape[1]**2, A.shape[3]**2, A.shape[2]**2)

    C = C / C.abs().max()
    T = T / T.abs().max()

    with torch.no_grad():
        spectrum = torch.linalg.eigvalsh(C).abs().sort(descending=True).values
        previous_spectrum = spectrum / spectrum.max()

    for i in range(ctm_max_iter):
        # ctm_iter
        # enlarge corner
        enlarged_C = torch.einsum("ab,bcd,eaf,dfgh->egch", C, T, T, E)
        left_chi, left_D, right_chi, right_D = enlarged_C.shape
        matrix = enlarged_C.reshape(left_chi * left_D, right_chi * right_D)
        # Remove roundoff asymmetry before diagonalizing.
        matrix = (matrix + matrix.mT) / 2
        # diagonalize and sort
        S, U = torch.linalg.eigh(matrix)
        indices = torch.argsort(S.abs(), descending=True)[:chi]
        S = S[indices]
        # get isometry
        U = U[:, indices].reshape(left_chi, left_D, len(indices))

        # absorb isometry into corner and edge tensors
        C = torch.diag(S)
        T = torch.einsum(T, (0, 1, 2), U, (1, 3, 4), E, (3, 2, 5, 6), U.conj(), (0, 5, 7), (7, 4, 6))

        # normalize corner and edge tensors and symmetrize
        C = C / C.abs().max()
        T = (T + T.permute(1, 0, 2)) / 2
        T = T / T.abs().max()

        # Compute the corner spectrum difference
        with torch.no_grad():
            spectrum = C.diagonal().abs().sort(descending=True).values
            # Boundary dimensions may grow during the first few updates.
            size = max(spectrum.numel(), previous_spectrum.numel())
            current_padded = torch.nn.functional.pad(
                spectrum, (0, size - spectrum.numel())
            )
            previous_padded = torch.nn.functional.pad(
                previous_spectrum, (0, size - previous_spectrum.numel())
            )
            residual = (current_padded - previous_padded).abs().sum()
            converged = bool(residual < ctm_conv_tol)
            previous_spectrum = spectrum

        if callback is not None:
            finished = converged or i == ctm_max_iter - 1
            callback(i, residual, finished, A, C, T)
        if converged:
            break

    if raise_on_nonconvergence and not converged:
        raise RuntimeError(
            f"CTMRG did not converge in {ctm_max_iter} iterations: "
            f"residual={residual.item():.3e}, tolerance={ctm_conv_tol:.3e}."
        )
    return C, T

## II.1 The spin-2 AKLT state

AKLT state is an example of non-trivial gapped paramagnet in two dimensions. In addition to internal SU(2)-symmetry it also retains lattice symmetries and cannot be adiabatically connected to the atomic limit. 
It is constructed by a direct extension of Spin-1 AKLT construction to regular two-dimensional lattice.
Here, we consider a variant on square lattice. It admits a very simple (i)PEPS representation.

**Goals**

- Build a square-lattice spin S=2 AKLT state from virtual singlets and local linear maps.
- Work out a variation of AKLT S=2 iPEPS generated by $C_{4v}$-symmetric on-site tensor (and sublattice rotation).
- Find it's parent Hamiltonian and evaluate it, together with other observables one the ground state.

The Affleck–Kennedy–Lieb–Tasaki (**AKLT**) construction i) assigns four virtual
spin-$\tfrac12$ degrees of freedom (DoFs) to each square-lattice site, ii) entangles neighboring virtual spins in
singlets, and iii) maps each group of four virtual spins onto one physical spin $S=2$.
The PEPS bond dimension is $D=2$; its physical dimension is $d=2S+1=5$.


Reference: [Mambrini, Orús, and Poilblanc, *Systematic construction of spin liquids
on the square lattice from tensor networks with SU(2) symmetry*](https://arxiv.org/abs/1608.06003),
Sec. II A and Appendix B.2, for the staggered-frame convention and tensor coefficients.

### II.1.0 Pre-requisites: Spin irreps

In [ ]:
from types import SimpleNamespace
from math import sqrt
import numpy as np
import matplotlib.pyplot as plt

torch.set_num_threads(2)


def close(actual, expected, atol=1e-12):
    expected = torch.as_tensor(expected, dtype=actual.dtype, device=actual.device)
    torch.testing.assert_close(actual, expected, atol=atol, rtol=0)

# Provided: spin irreps

# Spin-2 operators, with S^z in descending m order.
m_values = torch.arange(2., -3., -1.)
Splus = torch.zeros((5, 5))
for s in range(1, 5):
    m = m_values[s]
    Splus[s - 1, s] = torch.sqrt(6 - m * (m + 1))
Sminus = Splus.T.contiguous()
irrepS2= SimpleNamespace(Sz= torch.diag(m_values), Sx= (Splus + Sminus) / 2, Sy= (Splus - Sminus).to(torch.complex128) / (2j), 
                         Splus=Splus, Sminus=Sminus)

S12plus = torch.tensor([[0., 1.], [0., 0.]])
S12minus = S12plus.T.contiguous()
irrepShalf= SimpleNamespace(Sz= torch.diag(torch.tensor([1,-1]))/2.0, Sx= (S12plus + S12minus) / 2, Sy= (S12plus - S12minus).to(torch.complex128) / (2j), 
                            Splus=S12plus, Sminus=S12minus)

for name, op in irrepS2.__dict__.items():
    print(f"S2 {name}:\n{op}\n")
for name, op in irrepShalf.__dict__.items():
    print(f"Shalf {name}:\n{op}\n")

### II.1.1 Virtual state - product state of virtual singlets on each bond

We start with a square lattice, hosting 4 spin-1/2 DoFs (o) on each site

```text
  o     o        o         o 
 o o   o o  =>  o r -->-- l o   where r and l are entangled in a singlet
  o     o        o         o

  o     o        o         o
 o o   o o      o o       o o
  o     o        o         o
```

We want to pair the spin-1/2 DoFs up in singlets to create $|\Omega\rangle = \prod_{b\in\mathrm{bonds}} |singlet\rangle_b$.

Using $|0\rangle=|\uparrow\rangle$ and $|1\rangle=|\downarrow\rangle$, we entangle two virtual spins r,l
in a singlet 

$$
|\omega\rangle=(|0_r\rangle,|1_r\rangle)\omega(|0_l\rangle,|1_l\rangle)^T = \frac{|0_r1_l\rangle-|1_r0_l\rangle}{\sqrt2}
$$

Crucially, the singlet has an orientation, from `r` to `l`, here choosen by -1 factor in front of term with $|0_l\rangle$.
A $-|\omega\rangle$ correspond to the opposite orientation.

**Exercise 1:** return the normalized coefficient matrix `omega[a,b]`.

In [ ]:
def virtual_singlet():
    """Normalized singlet coefficients, with the even endpoint first."""
    # TODO 1: Fill the |01> and |10> coefficients and normalize the state.
    raise NotImplementedError("Exercise 1: construct the virtual singlet.")


omega = virtual_singlet()


### II.1.2 Map four virtual spins onto physical spin 2

At a site the four virtual spins span a $2^4=16$ dimensional space.
Its **fully symmetric** subspace has dimension five and carries spin 2.
Create a linear map $P$ 

$$
P: \otimes_4 \mathcal{H}_{S=1/2} \rightarrow \mathcal{H}_{S=2}
$$

which maps states from the product space of 4 spin-1/2s to its S=2 subspace.

Throughout, the leg order is **`(up,left,down,right,physical)`**:

```text
                     u(0), spin 1/2
                         |
  l(1), spin 1/2 -------[P]------- r(3), spin 1/2  =  P^s_uldr
                       / |
    s(4), spin 2 -----/  |
                         d(2), spin 1/2
```

**Exercise 2:** construct AKLT's on-site tensor $P$

In [ ]:
# Function to lift a sum of four operators, each acting on individual spin-1/2 spaces to their product space
def op_to_space_of_4halfs(o1, o2, o3, o4)->torch.Tensor:
    assert o1.shape[0] == o2.shape[0] == o3.shape[0] == o4.shape[0]
    I= torch.eye(o1.shape[0])
    return torch.kron(torch.kron(torch.kron(o1, I), I), I) + torch.kron(torch.kron(torch.kron(I, o2), I), I) \
        + torch.kron(torch.kron(torch.kron(I, I), o3), I) + torch.kron(torch.kron(torch.kron(I, I), I), o4)

def P_aklt():
    """Projector on maximally symmetric subspace of 4 spin-1/2 DoFs"""
    # Initialize projection matrix P. The state with maximal spin is the first column
    P= torch.zeros(2**4,5) 
    P[0,0]= 1.0
    
    # TODO 2: Create remaining states by application of the lowering operator S^- = \sum_{x\in\{u,l,d,r\}} S^-_x
    P[:,1]= op_to_space_of_4halfs(irrepShalf.Sminus, irrepShalf.Sminus, irrepShalf.Sminus, irrepShalf.Sminus) @ P[:,0]
    # ... the remaining states ...
       
    # TODO 3: These state spanning S=2 irrep are not normalized. Normalize each one (column).
    
    return P.reshape(2, 2, 2, 2, 5)

### II.1.3 Verify $C_{4v}$ symmetry

$C_{4v}$ is the symmetry group of a square: four rotations and four reflections.
The physical leg stays fixed while lattice operations permute `(u,l,d,r)`.
For example, a quarter turn sends `uldr` to `ldru`, and left/right reflection
sends it to `urdl`.

Our amplitudes depend only on the number of down spins, so they are invariant
under **all** permutations of the virtual legs, including $C_{4v}$.
This is the trivial, or $A_1$, representation. We verify it directly.

In [ ]:
A= P_aklt()
for permutation in [(1, 2, 3, 0, 4), (3, 0, 1, 2, 4), (2, 1, 0, 3, 4), (0, 3, 2, 1, 4)]:
    torch.testing.assert_close(A, A.permute(permutation))
print("Square-lattice symmetries verified.")

### II.1.4 From virtual singlet bonds to a C4v iPEPS

The AKLT state is obtained by applying $P$ on every site of the singlet network:

$$
|\Psi_{\rm AKLT}\rangle\propto
\left(\bigotimes_I P_I\right)
\left(\bigotimes_{\langle ij\rangle}|\omega\rangle_{ij}\right).
$$

```text
   Four virtual spins per site      Project onto spin 2

       o                 o                  |s0>           |s1>
       |                 |                   |              |
   o--( )--o =>=ω=>= o--( )--o     ->       [P]--->-ω->----[P]  i.e. P^{s0}_uldr ω_rL P^{s1}_ULDR
       |                 |                   |              |
       o                 o                  ...            ...

   =>=ω=>= joins two virtual spin-1/2 DoFs into a singlet.
   Each ( ) groups four virtual spins; P maps them to one physical spin.
```

An ordinary repeated-index contraction (Einstein summation convention) is realized via the identity bond matrix (or Kronecker delta) instead

$$
|EP\rangle = (|0_r\rangle,|1_r\rangle)\eta(|0_l\rangle,|1_l\rangle)^T = \frac{1}{\sqrt{2}}(|0_l0_r\rangle+|1_l1_r\rangle)
$$

where $\eta_{ab}=\delta_{ab}/\sqrt{2}$. Entangled pairs are symmetric wrt. to permutation of indices. 
To build a C4v-symmetric and uniform representation of AKLT states we make a specific choice of singlet orientations

```text
     -ω        ω
      ^        V
      o        o    
<-ω< obo >ω>  oao <-ω<
      o        o
      V        ^
      ω       -ω
      V        ^ 
      o        o    
 >ω> oao <-ω< obo >ω> 
      o        o  
      ^        V
     -ω        ω          a,b denote sublattice
```

i.e. singlets always point to sites of only one of the sublattices, say sublattice $a$.

### II.1.5 AKLT = Unitary x C4v PEPS

Let $r=e^{-i\pi S^y}$ act on a virtual spin S=1/2 and $R=e^{-i\pi S^y}$ act on a physical spin S=2. Crucially:

$$
r=\begin{pmatrix}0&-1\\1&0\end{pmatrix},\qquad
R_{s's}=(-1)^s\delta_{s',4-s}.
$$

The projection intertwines these rotations, $RP=P r^{\otimes4}$, and
$(I\otimes r)|EP\rangle=|\omega\rangle$.
Thus physical tensors $P$ on $b$ sites and $P'=RP$ on $a$ sites, joined by
ordinary bonds, reproduce the singlet construction (up to an overall scalar).
Both tensors remain $C_{4v}$ symmetric.

```text
 Physical AKLT (ordinary bonds)       Uniform tensor on oriented singlet virtual state

      P --- P' ---P                         P ->- P -<- P
      |     |     |                         V     ^     V
      P' -- P --- P'   <--- R on b ---      P -<- P ->- P
      |     |     |                         ^     V     ^
      P --- P' ---P                         P ->- P -<- P

P' = R P.
```

Contracting uniform $P$ therefore gives a state related to physical AKLT by $R$
on every $a$-sublattice site. Appreciate that $R$ is just a unitary transform.

**Exercise 3a:** contract $R$ onto the physical index to construct $P'$.

**Exercise 3b:** absorb $r$ on all four virtual inputs and recover the same $P'$.

In [ ]:
import math
r = torch.linalg.matrix_exp(-1j * math.pi * irrepShalf.Sy ).real
R = torch.linalg.matrix_exp(-1j * math.pi * irrepS2.Sy ).real

def physical_rotation(A, R):
    """Apply a physical rotation, keeping legs (u,l,d,r,s)."""
    # TODO 3a: Contract R[s,t] with the physical index t of P.
    raise NotImplementedError("Exercise 3a: rotate the physical spin.")


def virtual_rotations(A, r):
    """Absorb r[local_virtual, bond_index] on each virtual input."""
    # TODO 3b: Attach one copy of r to each of the four virtual indices of P.
    raise NotImplementedError("Exercise 3b: absorb the four virtual rotations.")

A= P_aklt()
assert torch.allclose(physical_rotation(A, R), virtual_rotations(A, r))

## II.2 An exact local parent-Hamiltonian

Two neighboring physical spins each have $S=2$, so their total spin can be
$J=0,1,2,3,4$ (again from decomposition of tensor product into direct sum of irreps). 
Their shared virtual singlet contributes zero spin, leaving
six virtual spin-$\tfrac12$ objects. These can have at most $J=3$.
Consequently the AKLT pair has **no $J=4$ component**.

Thus, the positive parent Hamiltonian is a projector on this $J=4$ subspace 

$$
H=\sum_{\langle ij\rangle}P_{J=4}^{ij}.
$$

Write $X=\mathbf S_i\cdot\mathbf S_j$. Its eigenvalues are $[J(J+1)-12]/2=-6,-5,-3,0,4$.
Consider a polynomial in $X$ which is one on the $X=4$ eigenspace and annihilates (is zero) on all other four eigenspaces

$$
P_{J=4}=\frac{X(X+3I)(X+5I)(X+6I)}{2520}
$$

**Exercise 4**: define parent Hamiltonian term $P_{J=4}$ 

The following provided check keeps all six external virtual spins open.
It is an exact local tensor identity and needs no infinite environment.

In [ ]:
# Provided: parent projector in the physical, unrotated spin basis.
I25 = torch.eye(25)
spin_dot = torch.kron(irrepS2.Sz, irrepS2.Sz) + torch.kron(irrepS2.Sx, irrepS2.Sx) + torch.kron(irrepS2.Sy, irrepS2.Sy).real

# TODO 4: Define local Hamiltonian for the AKLT model.
def local_hamiltonian():
    """Define the local Hamiltonian for the AKLT model."""
    raise NotImplementedError("Exercise 3b: absorb the four virtual rotations.")
    return P4

P4= local_hamiltonian()
torch.allclose(P4, P4.T)
torch.allclose(P4 @ P4, P4)

# Left tensor legs: (i,j,k,x,s); right tensor: (a,y,b,c,t).
# x--omega--y is their shared bond; i,j,k,a,b,c remain open.
pair = torch.einsum("ijkxs,xy,aybct->ijkabcst", A, omega, A)
pair_map = pair.reshape(64, 25).T
torch.allclose(P4 @ pair_map, torch.zeros_like(pair_map))
print("The local AKLT pair is annihilated by P(J=4).")

## II.3 Compute observables

### III.3.1 CTMRG environments

Provided `ctmrg` operates directly on PEPS tensor, by first constructing the double-layer tensor

$$
(E)_{(uU)(lL)(dD)(rR)}
=\sum_{s,t}A_{uldrs}\,\overline{A_{ULDRs}}.
$$

and recycling the CTMRG from Ising Exercise. Note, that whether we consider $A= P$ or $A= RP$ with $R$ unitary, the double-layer
tensor $E$ stays the same. Therefore, we can recycle Ising's ctmrg. The environment tensors $C$ and $T$ are independent of $R$.

### III.3.2 Energy and magnetization

Use `ctmrg` from the prerequisites, passing the rank-five PEPS tensor `A`.
It builds `E` internally and closes missing virtual ket/bra pairs with the identity,
giving a deterministic free virtual boundary. It returns `C, T`; the callback
reports the iteration count and residual.
Convergence of the corner spectrum controls iteration error; comparing several
$\chi$ values probes truncation error. A dense cutoff can split spin multiplets,
so check isotropy rather than assuming exact symmetry at finite $\chi$.

In [ ]:
# Provided: convergence monitor and default infinite environment.
def report_ctm(i, residual, finished, A, C, T):
    if finished:
        print(f"iterations={i + 1}; corner residual={residual.item():.2e}")


C, T = ctmrg(A, chi=24, ctm_max_iter=500, ctm_conv_tol=1e-8,
             callback=report_ctm, raise_on_nonconvergence=True)
print("chi=24; C:", tuple(C.shape), "; T:", tuple(T.shape))
print(f"spec(C) {C.diag().sort(descending=True).values}")

### III.3.3 Measure local physical spins and parent energy

Keep physical ket and bra indices open to obtain the one-site and two-site
reduced density matrices (RDMs). Their normalization is their trace.
We store $\rho_1[s,t]$ and $\rho_2[s_1,s_2,t_1,t_2]$, with all ket indices first.

```text
    C--T--C             C--T---T--C
    |  |  |             |  |   |  |
    T--AĀ--T            T--AĀ--AĀ--T
    |  |  |             |  |   |  |
    C--T--C             C--T---T--C

AĀ has physical s,t open. Each internal virtual line is a fused ket/bra pair.
```

These contractions use uniform $A$, hence return RDMs with action of unitary $R$.
For a pair whose first site is even, convert to the physical basis as
$\rho_2^{\rm phys}=(I\otimes R)\rho_2^{\rm C4v}(I\otimes R)^\dagger$.
Equivalently transform the operator by $O\mapsto(I\otimes R)^\dagger O(I\otimes R)$.

The spin-singlet state has $\rho_1=I_5/5$, zero magnetization, and
$\langle(S^\alpha)^2\rangle=S(S+1)/3=2$ for each axis.
Its parent-projector expectation is zero. The square lattice has two bonds
per site, so the parent energy per site is $2\langle P_{J=4}\rangle$.

**Exercise 5a:** Implement 2-site RDM contraction

**Exercies 5b:** Evaluate on-site magnetization

**Exercies 5c:** Evaluate energy per site

In [ ]:
def rdm_2site(A, C, T):
    """Nearest neighbors: rho[s1,s2,t1,t2] in the uniform/staggered frame.

    Construct the two open half strips independently, so no intermediate
    contains both physical-site pairs and two boundary indices.
    """
    # TODO 5a: Construct half of the two-site reduced density matrix.
    #         C--T---
    #         T--A---
    #         C--T---
    half = ...  # Construct the half-row using the tensors C, T, and A

    rdm = ... # multiply two halfs
    raise NotImplementedError("Exercise 5a: absorb the four virtual rotations.")

    # return normalized rdm
    return rdm/rdm.reshape(5**2,5**2).trace()

def Rrdm_2siteR(rdm):
    """Apply the R rotation to the two-site reduced density matrix."""
    # TODO 5b: Implement the rotation of the two-site reduced density matrix.
    raise NotImplementedError("Exercise 5b: apply the R rotation to the rdm.")
    return rdm

# TODO 5c: Evaluate energy per site and magnetization
print("Energy per site")
# ...
print("Magnetization per site as (<Sz>, <Sx>, <Sy>)")
# ...

## 8. Variational iPEPS optimization

Here we use the spin-2 Heisenberg model as an example to demonstrate the workflow of variational iPEPS optimizations. The Hamiltonian is written as

  $$
  H=\sum_{\langle ij\rangle}\mathbf S_i\cdot\mathbf S_j,
  $$
where set the coupling to $J=1$.

### 8.1. Initialize one variational tensor

- Use the checkerboard pattern from section 3, with **only $A$ varied**:
  $$
  B_{uldrs}=\sum_t R_{st}A_{uldrt},\qquad R=e^{-i\pi S^y}.
  $$
- Since $R^\dagger R=I$, both sublattices have the same norm layer. One set of CTM environment tensors suffices.

In [ ]:
torch.manual_seed(7)
D_opt, chi_opt = 2, 16

def symmetric_tensor(A):
    """Symmetrize A by applying the all the transformations in c4v."""
    site = sum(A.permute(*perm, 4) for perm in sorted(c4v)) / len(c4v)
    return site / site.norm()


data = torch.randn(D_opt, D_opt, D_opt, D_opt, 5)
A = symmetric_tensor(data).requires_grad_()


### 8.2. Define the energy

- `rdm_2site(A, C, T)` returns the normalized two-site RDM in the uniform, staggered frame.
- Rotate the bond Hamiltonian into this frame:

  $$
  h_{\mathrm{frame}}=(I\otimes R)^\dagger
  (\mathbf S_i\cdot\mathbf S_j)(I\otimes R)
  =-S^x\otimes S^x+S^y\otimes S^y-S^z\otimes S^z.
  $$

- All nearest-neighbor bonds have the same energy in this ansatz. There are two bonds per square-lattice site:

  $$
  e=2\,\mathrm{Tr}(\rho_{ij}^{\mathrm{frame}}h_{\mathrm{frame}}).
  $$

- Keep every contraction in PyTorch so the energy remains differentiable through both the RDM and CTMRG.


In [ ]:
U_pair = torch.kron(torch.eye(5), R)
h_frame = U_pair.mH @ spin_dot @ U_pair

def energy_per_site(A, C, T):
    """Contract the two-site RDM with the bond Hamiltonian; count two bonds per site."""
    rho = rdm_2site(A, C, T).reshape(25, 25)
    return 2 * torch.einsum("st,ts->", rho, h_frame)


### 8.3. Optimize with L-BFGS

- Give [L-BFGS](https://docs.pytorch.org/docs/stable/generated/torch.optim.LBFGS.html) the single parameter `A`.
- Each closure symmetrizes the tensor, converges a fresh environment with the CTMRG from the beginning, and evaluates the energy.


In [ ]:
optimizer = torch.optim.LBFGS(
    [A], max_iter=1, max_eval=25, line_search_fn="strong_wolfe"
)
ctm_options = dict(chi=chi_opt, ctm_max_iter=300, ctm_conv_tol=1e-8,
                   raise_on_nonconvergence=False)


def closure():
    """Return the current energy and populate A.grad."""
    optimizer.zero_grad()

    # symmetrize A
    site = symmetric_tensor(A)
    C_opt, T_opt = ctmrg(site, **ctm_options)
    energy = energy_per_site(site, C_opt, T_opt)

    # Backpropagation
    energy.backward()
    return energy


energies = []
for step in range(30):
    energy = optimizer.step(closure)  # Energy before this update.
    energies.append(energy.item())
    print(f"Step {step + 1:2d}: starting energy = {energy.item():.8f}")


In [ ]:
with torch.no_grad():
    A_opt = symmetric_tensor(A)
    B_opt = physical_rotation(A_opt, R)
    C_opt, T_opt = ctmrg(A_opt, **ctm_options)
    final_energy = energy_per_site(A_opt, C_opt, T_opt).item()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(len(energies) + 1), energies + [final_energy], "o-", ms=3)
ax.plot(len(energies), final_energy, "o", label="Final state")
ax.axhline(-8, color="gray", ls="--", label="Néel product state")
ax.set(xlabel="Optimization step", ylabel="Energy per site")
ax.legend()
fig.tight_layout()
plt.show()


The same workflow can be used to explore the interpolation between the Heisenberg and AKLT Hamiltonians:

$$
H(g)=(1-g)H_{\mathrm{Heisenberg}}+gH_{\mathrm{AKLT}}.
$$

An intesting question is to locate the quantum phase transition and investigate its nature.
